In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

Mounted at /content/drive


In [2]:
FILE_PATH = '/content/drive/MyDrive/DVA_Project/data/interim/eda_ready.csv'

df = pd.read_csv(FILE_PATH)

print(df.shape)
df.head()

(11971, 13)


,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,year,month
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,2024,4
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,2023,7
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,2022,10
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,False,2022,5
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,2022,10


In [3]:
print(df.dtypes)
print("\nMissing:\n", df.isnull().sum())

transaction_id       object
customer_id          object
category             object
item                 object
price_per_unit      float64
quantity            float64
total_spent         float64
payment_method       object
location             object
transaction_date     object
discount_applied       bool
year                  int64
month                 int64
dtype: object

Missing:
 transaction_id      0
customer_id         0
category            0
item                0
price_per_unit      0
quantity            0
total_spent         0
payment_method      0
location            0
transaction_date    0
discount_applied    0
year                0
month               0
dtype: int64


In [4]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

numeric_cols = ['price_per_unit', 'quantity', 'total_spent']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [5]:
df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month

In [6]:
def check_normality(series):
    sample = series.dropna()
    sample = sample.sample(min(len(sample), 5000), random_state=42)
    stat, p = stats.shapiro(sample)
    return p

def check_variance(*groups):
    stat, p = stats.levene(*groups)
    return p

def cohens_d(a, b):
    a, b = a.dropna(), b.dropna()
    pooled_std = np.sqrt(((a.std()**2) + (b.std()**2)) / 2)
    return (a.mean() - b.mean()) / pooled_std

def eta_squared(anova_table):
    ss_between = anova_table['sum_sq'][0]
    ss_total = anova_table['sum_sq'].sum()
    return ss_between / ss_total

In [7]:
group_true = df[df['discount_applied'] == True]['total_spent']
group_false = df[df['discount_applied'] == False]['total_spent']

# Assumptions
print("Normality p-values:",
      check_normality(group_true),
      check_normality(group_false))

print("Variance p-value:",
      check_variance(group_true, group_false))

# t-test
t_stat, p_val = stats.ttest_ind(group_true, group_false, equal_var=False)
print("T-test p-value:", p_val)

# Effect size
print("Cohen's d:", cohens_d(group_true, group_false))

Normality p-values: 2.2856950974512402e-41 4.391359456788273e-45
Variance p-value: 0.37657120644879094
T-test p-value: 0.49305273511281056
Cohen's d: 0.013294050129216115


In [8]:
groups = [group['total_spent'].values for name, group in df.groupby('category')]

f_stat, p_val = stats.f_oneway(*groups)
print("ANOVA p-value (category):", p_val)

ANOVA p-value (category): 9.203823006774923e-07


In [9]:
groups = [group['total_spent'].values for name, group in df.groupby('payment_method')]

f_stat, p_val = stats.f_oneway(*groups)
print("ANOVA p-value (payment_method):", p_val)

ANOVA p-value (payment_method): 0.4966004858106826


In [10]:
groups = [group['total_spent'].values for name, group in df.groupby('location')]

f_stat, p_val = stats.f_oneway(*groups)
print("ANOVA p-value (location):", p_val)

ANOVA p-value (location): 0.3676788111085655


In [11]:
groups = [group['total_spent'].values for name, group in df.groupby('month')]

f_stat, p_val = stats.f_oneway(*groups)
print("ANOVA p-value (month):", p_val)

ANOVA p-value (month): 0.346281987067072


In [12]:
corr_price, p_price = stats.pearsonr(df['price_per_unit'], df['total_spent'])
corr_qty, p_qty = stats.pearsonr(df['quantity'], df['total_spent'])

print("Price vs Total:", corr_price, p_price)
print("Quantity vs Total:", corr_qty, p_qty)

Price vs Total: 0.6306161535368676 0.0
Quantity vs Total: 0.7120690726226545 0.0


In [13]:
def confidence_interval(series):
    mean = np.mean(series)
    sem = stats.sem(series)
    interval = stats.t.interval(0.95, len(series)-1, loc=mean, scale=sem)
    return interval

for col in ['category', 'payment_method']:
    print(f"\nConfidence intervals for {col}:")
    for name, group in df.groupby(col):
        ci = confidence_interval(group['total_spent'])
        print(name, ci)


Confidence intervals for category:
Beverages (np.float64(126.72217688185074), np.float64(136.7103097491653))
Butchers (np.float64(134.0273577432403), np.float64(144.20526257761534))
Computers And Electric Accessories (np.float64(124.58159450002009), np.float64(133.63438383444165))
Electric Household Essentials (np.float64(129.62445536947564), np.float64(139.25879001311012))
Food (np.float64(124.69351647447498), np.float64(133.84928379095305))
Furniture (np.float64(123.1361697794768), np.float64(133.00809251560514))
Milk Products (np.float64(114.35958139829614), np.float64(123.72634061095701))
Patisserie (np.float64(121.70501025219144), np.float64(131.12705081651086))

Confidence intervals for payment_method:
Cash (np.float64(128.1518682905759), np.float64(133.95390797069635))
Credit Card (np.float64(126.14784920304264), np.float64(132.10628881580124))
Digital Wallet (np.float64(125.77486459249002), np.float64(131.66182660263814))


In [14]:
print("Mean total_spent by category:")
print(df.groupby('category')['total_spent'].mean().sort_values(ascending=False))

print("\nMean by payment_method:")
print(df.groupby('payment_method')['total_spent'].mean())

Mean total_spent by category:
category
Butchers                              139.116310
Electric Household Essentials         134.441623
Beverages                             131.716243
Food                                  129.271400
Computers And Electric Accessories    129.107989
Furniture                             128.072131
Patisserie                            126.416031
Milk Products                         119.042961
Name: total_spent, dtype: float64

Mean by payment_method:
payment_method
Cash              131.052888
Credit Card       129.127069
Digital Wallet    128.718346
Name: total_spent, dtype: float64


In [15]:
print(df.shape)
print(df.isnull().sum())

(11971, 13)
transaction_id      0
customer_id         0
category            0
item                0
price_per_unit      0
quantity            0
total_spent         0
payment_method      0
location            0
transaction_date    0
discount_applied    0
year                0
month               0
dtype: int64


In [16]:
OUTPUT_PATH = '/content/drive/MyDrive/DVA_Project/data/interim/stat_ready.csv'
df.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)

Saved to: /content/drive/MyDrive/DVA_Project/data/interim/stat_ready.csv
